# Лабораторная работа 2

Подбор модели по `train_data.pickle`, сохранение лучшей модели и проверка CLI для Docker.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


In [ ]:
DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train_data.pickle'
TEST_PATH = DATA_DIR / 'test_data.pickle'
MODEL_PATH = DATA_DIR / 'model_best.pickle'

with open(TRAIN_PATH, 'rb') as file:
    train_data = pickle.load(file)

X = train_data['X']
y = train_data['y']


In [ ]:
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print('Target values and counts:', dict(zip(*np.unique(y, return_counts=True))))
print('NaN in X:', np.isnan(X).sum())
print('NaN in y:', np.isnan(y).sum())


Целевая переменная принимает три дискретных значения `0`, `1`, `2`, значит решается задача многоклассовой классификации.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'f1_weighted': 'f1_weighted',
    'precision_weighted': 'precision_weighted',
    'recall_weighted': 'recall_weighted',
}

models = {
    'SVC': Pipeline([
        ('scaler', StandardScaler()),
        ('model', SVC(class_weight='balanced', random_state=42)),
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('model', KNeighborsClassifier()),
    ]),
    'RandomForest': RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced',
    ),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
}


In [ ]:
rows = []
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=1)
    row = {'model': name}
    for metric in scoring:
        values = scores[f'test_{metric}']
        row[f'{metric}_mean'] = values.mean()
        row[f'{metric}_std'] = values.std()
    rows.append(row)

results = (
    pd.DataFrame(rows)
    .sort_values('f1_weighted_mean', ascending=False)
    .reset_index(drop=True)
)
results


В качестве основной метрики выбрана `f1_weighted`: классы не идеально сбалансированы, а метрика учитывает качество по всем классам с весами по поддержке.

In [ ]:
svc_param_grid = {
    'model__C': [0.1, 1, 3, 10],
    'model__kernel': ['linear', 'rbf'],
    'model__gamma': ['scale', 'auto'],
}

svc_grid = GridSearchCV(
    estimator=models['SVC'],
    param_grid=svc_param_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=1,
)
svc_grid.fit(X, y)

print('Best CV f1_weighted:', round(svc_grid.best_score_, 4))
print('Best params:', svc_grid.best_params_)


In [ ]:
best_model = svc_grid.best_estimator_
best_model.fit(X, y)

with open(MODEL_PATH, 'wb') as file:
    pickle.dump(best_model, file, protocol=4)

print(f'Model saved to {MODEL_PATH}')


In [ ]:
with open(TEST_PATH, 'rb') as file:
    test_data = pickle.load(file)

X_test = test_data['X']
y_test = test_data['y']
y_pred = best_model.predict(X_test)

test_metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred, average='weighted'),
    'precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
    'recall': recall_score(y_test, y_pred, average='weighted'),
}

test_metrics
